In [ ]:
!pip install tensorflow
!pip install keras-tuner -q

In [ ]:

# Step 1: IMPORTS AND SETUP

import os
import glob
import numpy as np
import pandas as pd
import pickle


import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.regularizers import l2
import keras_tuner as kt


from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder




# Step 2: CONFIGURATION

DATA_DIR     = "/content/drive/MyDrive/relabelled_dataset_new/Data"
FILE_GLOB    = "*.csv"
OUT_DIR      = "/content/drive/MyDrive/relabelled_dataset_new/Model/Model_Hybrid_CNN_1"
os.makedirs(OUT_DIR, exist_ok=True)


COLS_AXES    = ["ax", "ay", "az"]
FS           = 2720.0
WIN          = 128
HOP          = 64
F0, DELTA    = 85.0, 25.0


FEATURES_KEEP = [
    "jerk_rms",
    "centroid_norm",
    "zcr",
    "crest_factor_amp",
    "ber_f0",
    "high_ratio",
    "bandpower_over_entropy_amp"
]
NUM_FEATURES = len(FEATURES_KEEP)



# Step 3: FEATURE EXTRACTION FUNCTIONS

# --- helpers ---
def read_xyz_csv(path, cols=COLS_AXES):
    df = pd.read_csv(path)
    if not all(c in df.columns for c in cols):
        num_cols = [c for c in df.columns if np.issubdtype(df[c].dtype, np.number)]
        arr = df[num_cols[:3]].to_numpy(dtype=np.float32)
    else:
        arr = df[cols].to_numpy(dtype=np.float32)
    return arr

def sliding_starts(T, win, hop):
    i=0; idx=[]
    while i+win <= T:
        idx.append(i); i += hop
    return idx

# --- feature calculation functions ---
def goertzel_pow(x, f, fs):
    N = len(x); k = int(0.5 + (N*f)/fs)
    w = (2*np.pi/N)*k
    cw, sw = np.cos(w), np.sin(w)
    s1 = 0.0; s2 = 0.0
    for n in range(N):
        s0 = x[n] + 2*cw*s1 - s2
        s2, s1 = s1, s0
    real = s1 - s2*cw; imag = s2*sw
    return (real*real + imag*imag)/N

def spectral_entropy(x, fs):
    P = np.abs(np.fft.rfft(x - np.mean(x)))**2
    P = P / (np.sum(P) + 1e-12)
    H = -np.sum(P * np.log(P + 1e-12))
    return float(H / np.log(len(P)))

def zero_crossing_rate(x):
    xm = x - np.mean(x)
    return float(np.mean(np.abs(np.diff(np.signbit(xm)))))

def band_energy_ratio(x, fs, f_lo, f_hi):
    freqs = np.fft.rfftfreq(len(x), 1/fs)
    X = np.abs(np.fft.rfft(x - np.mean(x)))**2
    total = np.sum(X) + 1e-12
    band = (freqs>=f_lo) & (freqs<=f_hi)
    return float(np.sum(X[band]) / total)

def spectral_stats_basic(x, fs, f0=F0):
    freqs = np.fft.rfftfreq(len(x), 1/fs)
    S = np.abs(np.fft.rfft(x - x.mean())) + 1e-9
    peak_ratio   = float(np.max(S[1:]) / (np.sum(S) + 1e-9))
    centroid     = float(np.sum(freqs*S) / np.sum(S))
    centroid_norm= centroid/(fs/2.0)
    geo          = np.exp(np.mean(np.log(S)))
    flatness     = float(geo/np.mean(S))
    band = (freqs>=f0) & (freqs<=3*f0)
    high = freqs>=2.5*f0
    high_ratio   = float((np.sum(S[high])+1e-9) / (np.sum(S[band])+1e-9))
    return peak_ratio, centroid_norm, flatness, high_ratio

def dominant_amp_in_band(x, fs, f_lo, f_hi):
    freqs = np.fft.rfftfreq(len(x), 1/fs)
    X = np.abs(np.fft.rfft(x - np.mean(x)))
    band = (freqs>=f_lo) & (freqs<=f_hi)
    xb = X[band]
    return float(np.max(xb)) if xb.size else 0.0

def crest_factor(x):
    rms = np.sqrt(np.mean(x*x) + 1e-12)
    return float(np.max(np.abs(x)) / (rms + 1e-12))

def compute_selected_features(ax, ay, az, fs=FS):
    a = np.sqrt(ax*ax + ay*ay + az*az).astype(np.float32)
    pr, cen_n, flat, hr = spectral_stats_basic(a, fs)
    jerk = np.diff(a, prepend=a[0]) * fs
    jerk_rms = float(np.sqrt(np.mean(jerk*jerk)))
    zcr = zero_crossing_rate(a)
    ber_f0 = band_energy_ratio(a, fs, F0-10, F0+10)
    band_pow = band_energy_ratio(a, fs, F0-2*DELTA, F0+2*DELTA) * (np.sum(np.abs(np.fft.rfft(a - np.mean(a)))**2)+1e-12)
    dom_amp  = dominant_amp_in_band(a, fs, F0-2*DELTA, F0+2*DELTA)
    Hn = spectral_entropy(a, fs)
    bandpower_over_entropy_amp = band_pow / (Hn + 1e-12)
    dom_amp_over_bandpower_amp = dom_amp / (band_pow + 1e-12)
    cf = crest_factor(a)
    return { "jerk_rms": jerk_rms, "centroid_norm": cen_n, "zcr": zcr, "crest_factor_amp": cf,
        "ber_f0": ber_f0, "high_ratio": hr, "bandpower_over_entropy_amp": bandpower_over_entropy_amp,
        "dom_amp_over_bandpower_amp": dom_amp_over_bandpower_amp }


# Step 4: DATA LOADING AND PREPROCESSING

print("Starting data loading and feature extraction...")
all_windows, all_features, all_labels = [], [], []
class_dirs = sorted([d for d in glob.glob(os.path.join(DATA_DIR, "*")) if os.path.isdir(d)])
assert class_dirs, f"No class folders found under {DATA_DIR}"

for cdir in class_dirs:
    label = os.path.basename(cdir)
    print(f"Processing class: {label}")
    for fp in sorted(glob.glob(os.path.join(cdir, FILE_GLOB))):
        xyz = read_xyz_csv(fp)
        for s in sliding_starts(len(xyz), WIN, HOP):
            # The raw windowed data for the CNN branch
            seg = xyz[s:s+WIN, :]
            all_windows.append(seg)

            # The handcrafted features for the MLP branch
            feats = compute_selected_features(seg[:,0], seg[:,1], seg[:,2], fs=FS)
            all_features.append([feats[k] for k in FEATURES_KEEP])


            all_labels.append(label)

# Convert lists to NumPy arrays
X_windows = np.asarray(all_windows, dtype=np.float32)
X_features = np.asarray(all_features, dtype=np.float32)
y_str = np.array(all_labels, dtype=object)

print("\nData processing complete.")
print("Shape of raw window data (for CNN):", X_windows.shape)
print("Shape of feature data (for MLP):", X_features.shape)
print("Total labels collected:", len(y_str))
print("Classes:", sorted(np.unique(y_str)))

# --- Encode Labels and Split Data ---
le = LabelEncoder()
y = le.fit_transform(y_str)
N_CLASSES = len(le.classes_)
print("\nLabel order:", list(le.classes_))

# Split all data (windows, features, and labels) together
X_win_tr, X_win_tmp, X_feat_tr, X_feat_tmp, y_tr, y_tmp = train_test_split(
    X_windows, X_features, y, test_size=0.30, stratify=y, random_state=42
)
X_win_va, X_win_te, X_feat_va, X_feat_te, y_va, y_te = train_test_split(
    X_win_tmp, X_feat_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=42
)

# --- Scale Handcrafted Features ---

scaler = StandardScaler().fit(X_feat_tr)
X_feat_tr_s = scaler.transform(X_feat_tr)
X_feat_va_s = scaler.transform(X_feat_va)
X_feat_te_s = scaler.transform(X_feat_te)


y_tr_cat = tf.keras.utils.to_categorical(y_tr, num_classes=N_CLASSES)
y_va_cat = tf.keras.utils.to_categorical(y_va, num_classes=N_CLASSES)
y_te_cat = tf.keras.utils.to_categorical(y_te, num_classes=N_CLASSES)

print("\nDataset shapes after splitting and scaling:")
print(f"Train set:   Windows={X_win_tr.shape}, Features={X_feat_tr_s.shape}, Labels={y_tr_cat.shape}")
print(f"Val set:     Windows={X_win_va.shape}, Features={X_feat_va_s.shape}, Labels={y_va_cat.shape}")
print(f"Test set:    Windows={X_win_te.shape}, Features={X_feat_te_s.shape}, Labels={y_te_cat.shape}")



with open(os.path.join(OUT_DIR, "scaler.pkl"), "wb") as f: pickle.dump(scaler, f)
with open(os.path.join(OUT_DIR, "label_encoder.pkl"), "wb") as f: pickle.dump(le, f)
with open(os.path.join(OUT_DIR, "feature_order.txt"), "w") as f: f.write("\n".join(FEATURES_KEEP))
print(f"\nScaler and Label Encoder saved to {OUT_DIR}")



# Step 5: HYBRID MODEL DEFINITION FOR KERAS TUNER

def build_hybrid_model(hp):
    """Builds a tunable hybrid 1D-CNN + MLP model for time series."""
    """L2 regularization is also a hyperparameter"""


    hp_l2_rate = hp.Choice('l2_rate', values=[1e-2, 1e-3, 1e-4, 0.0])


    # --- 1D CNN Branch (for raw windowed data) ---
    # Input shape is (window_size, num_axes)
    cnn_input = layers.Input(shape=(WIN, len(COLS_AXES)), name="cnn_input")
    x = cnn_input

    # Tune the number of 1D convolutional layers (e.g., 1 to 3)
    for i in range(hp.Int('num_conv_layers', 1, 3)):
        x = layers.Conv1D(
            filters=hp.Int(f'filters_{i}', min_value=8, max_value=48, step=4),
            kernel_size=hp.Choice(f'kernel_{i}', values=[3, 5, 7]),
            activation='relu',
            padding='same',
             # --- APPLY THE TUNABLE REGULARIZER ---
            kernel_regularizer=l2(hp_l2_rate)
        )(x)

        x = layers.MaxPooling1D(pool_size=2)(x)

    x = layers.Flatten()(x)
    cnn_output = layers.Dense(
        units=hp.Int('cnn_dense_units', min_value=16, max_value=64, step=8),
        activation='relu',

            kernel_regularizer=l2(hp_l2_rate)
    )(x)


    # --- MLP Branch (for handcrafted features) ---
    feature_input = layers.Input(shape=(NUM_FEATURES,), name="feature_input")
    y = feature_input
    y = layers.Dense(
        units=hp.Int('feature_dense_units', min_value=8, max_value=48, step=8),
        activation='relu'
    )(y)
    feature_output = y


    # --- Merge and Final Classifier ---
    concatenated = layers.concatenate([cnn_output, feature_output])
    merged = layers.Dropout(rate=hp.Float('dropout_rate', 0.1, 0.5, step=0.1))(concatenated)
    merged = layers.Dense(
        units=hp.Int('final_dense_units', 16, 128, step=16),
        activation='relu',

            kernel_regularizer=l2(hp_l2_rate)
    )(merged)

    output = layers.Dense(N_CLASSES, activation='softmax')(merged)
    model = keras.Model(inputs=[cnn_input, feature_input], outputs=output)

    # --- Compile the Model ---
    hp_learning_rate = hp.Choice('learning_rate', values=[1e-2, 1e-3, 1e-4])
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=hp_learning_rate),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model


# Step 6: HYPERPARAMETER SEARCH

tuner = kt.Hyperband(
    build_hybrid_model,
    objective='val_accuracy',
    max_epochs=20,
    factor=3,
    directory=OUT_DIR,
    project_name='drill_grip_tuning'
)

stop_early = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5)

print("\nStarting hyperparameter search...")

tuner.search(
    [X_win_tr, X_feat_tr_s], y_tr_cat,
    epochs=50,
    validation_data=([X_win_va, X_feat_va_s], y_va_cat),
    callbacks=[stop_early]
)
print("Hyperparameter search complete.")



# Step 7: RETRIEVE, TRAIN, AND SAVE THE BEST MODEL

best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
print("\n--- Best Hyperparameters Found ---")
for hp_name in best_hps.values:
    print(f"{hp_name}: {best_hps.get(hp_name)}")

# Building final model with the optimal hyperparameters
best_model = tuner.hypermodel.build(best_hps)

print("\n--- Summary of the Best Model Architecture ---")
best_model.summary()

# Training best model on the combined training and validation data
X_win_full_train = np.concatenate((X_win_tr, X_win_va))
X_feat_full_train_s = np.concatenate((X_feat_tr_s, X_feat_va_s))
y_full_train_cat = np.concatenate((y_tr_cat, y_va_cat))

print("\n--- Training the Best Model on Full Training Data ---")
history = best_model.fit(
    [X_win_full_train, X_feat_full_train_s], y_full_train_cat,
    epochs=50,
    callbacks=[tf.keras.callbacks.EarlyStopping(monitor='loss', patience=10)]
)


print("\n--- Final Evaluation on Unseen Test Data ---")
test_loss, test_acc = best_model.evaluate([X_win_te, X_feat_te_s], y_te_cat)
print(f'\nTest accuracy: {test_acc:.4f}')


model_save_path = os.path.join(OUT_DIR, "best_hybrid_model.keras")
best_model.save(model_save_path)
print(f"Final model saved to {model_save_path}")


# Step 8: CONVERTING TO TFLITE WITH INT8 QUANTIZATION

print("\n--- Converting Model to TensorFlow Lite with INT8 Quantization ---")


def representative_data_gen():

  for i in range(300):

    yield [
        X_win_tr[i:i+1].astype(np.float32),
        X_feat_tr_s[i:i+1].astype(np.float32)
    ]


converter = tf.lite.TFLiteConverter.from_keras_model(best_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_data_gen

converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

tflite_model_int8 = converter.convert()


tflite_model_path = os.path.join(OUT_DIR, "model_int8.tflite")
with open(tflite_model_path, 'wb') as f:
    f.write(tflite_model_int8)

print(f"\nINT8 quantized TFLite model saved to: {tflite_model_path}")
print(f"Model size: {os.path.getsize(tflite_model_path) / 1024:.2f} KB")

# **Training**

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import pickle
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.regularizers import l2
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder


# Step 1: CONFIGURATION

DATA_DIR     = "/content/drive/MyDrive/relabelled_dataset_new/Data"
FILE_GLOB    = "*.csv"

OUT_DIR      = "/content/drive/MyDrive/relabelled_dataset_new/Model/Model_Hybrid_CNN_1"
os.makedirs(OUT_DIR, exist_ok=True)


BEST_HPS = {
    "l2_rate": 0.0001,
    "num_conv_layers": 3,
    "filters": [16, 48, 48],
    "kernels": [3, 5, 5],
    "cnn_dense_units": 32,
    "feature_dense_units": 24,
    "dropout_rate": 0.2,
    "final_dense_units": 48,
    "learning_rate": 0.001,
    "training_epochs": 100,
    "batch_size": 32
}

# --- Data Parameters ---
COLS_AXES    = ["ax", "ay", "az"]
FS           = 2750.0
WIN          = 128
HOP          = 64
F0, DELTA    = 95.0, 25.0

FEATURES_KEEP = [
    "jerk_rms", "centroid_norm", "zcr", "crest_factor_amp",
    "ber_f0", "high_ratio", "bandpower_over_entropy_amp"
]
NUM_FEATURES = len(FEATURES_KEEP)


# Step 2: FEATURE EXTRACTION FUNCTIONS

def read_xyz_csv(path, cols=COLS_AXES):
    df = pd.read_csv(path)
    if not all(c in df.columns for c in cols):
        num_cols = [c for c in df.columns if np.issubdtype(df[c].dtype, np.number)]
        arr = df[num_cols[:3]].to_numpy(dtype=np.float32)
    else:
        arr = df[cols].to_numpy(dtype=np.float32)
    return arr
def sliding_starts(T, win, hop):
    i=0; idx=[]
    while i+win <= T:
        idx.append(i); i += hop
    return idx
def spectral_entropy(x, fs):
    P = np.abs(np.fft.rfft(x - np.mean(x)))**2
    P = P / (np.sum(P) + 1e-12)
    H = -np.sum(P * np.log(P + 1e-12))
    return float(H / np.log(len(P)))
def zero_crossing_rate(x):
    xm = x - np.mean(x)
    return float(np.mean(np.abs(np.diff(np.signbit(xm)))))
def band_energy_ratio(x, fs, f_lo, f_hi):
    freqs = np.fft.rfftfreq(len(x), 1/fs)
    X = np.abs(np.fft.rfft(x - np.mean(x)))**2
    total = np.sum(X) + 1e-12
    band = (freqs>=f_lo) & (freqs<=f_hi)
    return float(np.sum(X[band]) / total)
def spectral_stats_basic(x, fs, f0=F0):
    freqs = np.fft.rfftfreq(len(x), 1/fs)
    S = np.abs(np.fft.rfft(x - x.mean())) + 1e-9
    peak_ratio   = float(np.max(S[1:]) / (np.sum(S) + 1e-9))
    centroid     = float(np.sum(freqs*S) / np.sum(S))
    centroid_norm= centroid/(fs/2.0)
    geo          = np.exp(np.mean(np.log(S)))
    flatness     = float(geo/np.mean(S))
    band = (freqs>=f0) & (freqs<=3*f0)
    high = freqs>=2.5*f0
    high_ratio   = float((np.sum(S[high])+1e-9) / (np.sum(S[band])+1e-9))
    return peak_ratio, centroid_norm, flatness, high_ratio
def crest_factor(x):
    rms = np.sqrt(np.mean(x*x) + 1e-12)
    return float(np.max(np.abs(x)) / (rms + 1e-12))
def compute_selected_features(ax, ay, az, fs=FS):
    a = np.sqrt(ax*ax + ay*ay + az*az).astype(np.float32)
    _, cen_n, _, hr = spectral_stats_basic(a, fs)
    jerk = np.diff(a, prepend=a[0]) * fs
    jerk_rms = float(np.sqrt(np.mean(jerk*jerk)))
    zcr = zero_crossing_rate(a)
    ber_f0 = band_energy_ratio(a, fs, F0-10, F0+10)
    band_pow = band_energy_ratio(a, fs, F0-2*DELTA, F0+2*DELTA) * (np.sum(np.abs(np.fft.rfft(a - np.mean(a)))**2)+1e-12)
    Hn = spectral_entropy(a, fs)
    bandpower_over_entropy_amp = band_pow / (Hn + 1e-12)
    cf = crest_factor(a)

    return { "jerk_rms": jerk_rms, "centroid_norm": cen_n, "zcr": zcr, "crest_factor_amp": cf,
        "ber_f0": ber_f0, "high_ratio": hr, "bandpower_over_entropy_amp": bandpower_over_entropy_amp }


# Step 3: DATA LOADING AND PREPROCESSING

print("--- Starting Data Loading and Preprocessing ---")
all_windows, all_features, all_labels = [], [], []
class_dirs = sorted([d for d in glob.glob(os.path.join(DATA_DIR, "*")) if os.path.isdir(d)])
assert class_dirs, f"No class folders found under {DATA_DIR}"

for cdir in class_dirs:
    label = os.path.basename(cdir)
    print(f"Processing class: {label}")
    for fp in sorted(glob.glob(os.path.join(cdir, FILE_GLOB))):
        xyz = read_xyz_csv(fp)
        for s in sliding_starts(len(xyz), WIN, HOP):
            seg = xyz[s:s+WIN, :]
            all_windows.append(seg)
            feats = compute_selected_features(seg[:,0], seg[:,1], seg[:,2], fs=FS)
            all_features.append([feats[k] for k in FEATURES_KEEP])
            all_labels.append(label)

X_windows = np.asarray(all_windows, dtype=np.float32)
X_features = np.asarray(all_features, dtype=np.float32)
y_str = np.array(all_labels, dtype=object)

print(f"\nTotal samples: {len(y_str)}")


le = LabelEncoder()
y = le.fit_transform(y_str)
N_CLASSES = len(le.classes_)
print(f"Classes ({N_CLASSES}): {list(le.classes_)}")

# 70% Train, 15% Val, 15% Test
X_win_tr, X_win_tmp, X_feat_tr, X_feat_tmp, y_tr, y_tmp = train_test_split(
    X_windows, X_features, y, test_size=0.30, stratify=y, random_state=42)
X_win_va, X_win_te, X_feat_va, X_feat_te, y_va, y_te = train_test_split(
    X_win_tmp, X_feat_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=42)

# --- Scale Features ---
scaler = StandardScaler().fit(X_feat_tr)
X_feat_tr_s = scaler.transform(X_feat_tr)
X_feat_va_s = scaler.transform(X_feat_va)
X_feat_te_s = scaler.transform(X_feat_te)


y_tr_cat = tf.keras.utils.to_categorical(y_tr, num_classes=N_CLASSES)
y_va_cat = tf.keras.utils.to_categorical(y_va, num_classes=N_CLASSES)
y_te_cat = tf.keras.utils.to_categorical(y_te, num_classes=N_CLASSES)


with open(os.path.join(OUT_DIR, "scaler.pkl"), "wb") as f: pickle.dump(scaler, f)
with open(os.path.join(OUT_DIR, "label_encoder.pkl"), "wb") as f: pickle.dump(le, f)
print(f"Data artifacts saved to {OUT_DIR}")


# Step 4: DEFINE AND TRAIN THE FIXED HYBRID MODEL

print("\n--- Building and Training the Fixed Hybrid Model ---")

def build_fixed_hybrid_model():
    # -- 1D CNN Branch --
    cnn_input = layers.Input(shape=(WIN, len(COLS_AXES)), name="cnn_input")
    x = cnn_input

    for i in range(BEST_HPS["num_conv_layers"]):
        x = layers.Conv1D(
            filters=BEST_HPS["filters"][i],
            kernel_size=BEST_HPS["kernels"][i],
            activation='relu', padding='same',
            kernel_regularizer=l2(BEST_HPS["l2_rate"])
        )(x)
        x = layers.MaxPooling1D(pool_size=2)(x)

    x = layers.Flatten()(x)
    cnn_output = layers.Dense(
        units=BEST_HPS["cnn_dense_units"], activation='relu',
        kernel_regularizer=l2(BEST_HPS["l2_rate"])
    )(x)

    # -- MLP Branch --
    feature_input = layers.Input(shape=(NUM_FEATURES,), name="feature_input")
    feature_output = layers.Dense(
        units=BEST_HPS["feature_dense_units"], activation='relu',
        kernel_regularizer=l2(BEST_HPS["l2_rate"])
    )(feature_input)

    # -- Merge and Classify --
    merged = layers.Concatenate()([cnn_output, feature_output])
    merged = layers.Dropout(rate=BEST_HPS["dropout_rate"])(merged)
    merged = layers.Dense(
        units=BEST_HPS["final_dense_units"], activation='relu',
        kernel_regularizer=l2(BEST_HPS["l2_rate"])
    )(merged)
    output = layers.Dense(N_CLASSES, activation='softmax')(merged)

    model = keras.Model(inputs=[cnn_input, feature_input], outputs=output)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=BEST_HPS["learning_rate"]),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model


model = build_fixed_hybrid_model()
model.summary()


X_win_train_full = np.concatenate((X_win_tr, X_win_va))
X_feat_train_full_s = np.concatenate((X_feat_tr_s, X_feat_va_s))
y_train_full_cat = np.concatenate((y_tr_cat, y_va_cat))

print("Starting training...")
history = model.fit(
    [X_win_train_full, X_feat_train_full_s], y_train_full_cat,
    epochs=BEST_HPS["training_epochs"],
    batch_size=BEST_HPS["batch_size"],

    validation_data=([X_win_te, X_feat_te_s], y_te_cat),
    callbacks=[
        tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5),
        tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)
    ],
    verbose=1
)


keras_model_path = os.path.join(OUT_DIR, "best_hybrid_model.keras")
model.save(keras_model_path)
print(f"\nTrained Keras model saved to: {keras_model_path}")


print("\n--- Final Evaluation on Test Set ---")
test_loss, test_acc = model.evaluate([X_win_te, X_feat_te_s], y_te_cat, verbose=0)
print(f"Test Accuracy: {test_acc:.4f}")


# Step 5: ROBUST TFLITE CONVERSION WITH INT8 QUANTIZATION

print("\n--- Converting to TFLite with Pure INT8 Quantization ---")


REP_DATA_SIZE = 300
X_win_rep = X_win_tr[:REP_DATA_SIZE]
X_feat_rep = X_feat_tr_s[:REP_DATA_SIZE]

def representative_data_gen():
    for i in range(len(X_win_rep)):
        2
        yield [X_win_rep[i:i+1].astype(np.float32), X_feat_rep[i:i+1].astype(np.float32)]

print("Creating concrete function...")
run_model = tf.function(lambda inputs: model(inputs))
input_specs = [tf.TensorSpec(spec.shape, spec.dtype) for spec in model.inputs]
concrete_func = run_model.get_concrete_function(input_specs)


print("Configuring converter...")
converter = tf.lite.TFLiteConverter.from_concrete_functions([concrete_func])
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_data_gen


converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]

converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8


print("Running conversion")
try:
    tflite_model_quantized = converter.convert()

    tflite_model_path = os.path.join(OUT_DIR, "model_int8.tflite")
    with open(tflite_model_path, 'wb') as f:
        f.write(tflite_model_quantized)

    print("-" * 50)
    print("SUCCESS! Conversion complete.")
    print(f"Final INT8 TFLite model saved to: {tflite_model_path}")
    print(f"Model size: {os.path.getsize(tflite_model_path) / 1024:.2f} KB")
    print("-" * 50)

except Exception as e:
    print(f"\nERROR during TFLite conversion:\n{e}")